# The REG201_PROC_STEP: procedural step table

Welcome to the ``REG201_PROC_STEP`` table overview. The ``REG201_PROC_STEP`` table contains essential procedural data that is part of the European Patent Register. While not all of this information is published in the European Patent Bulletin, it provides insights into various procedural steps throughout the patent application process. Each record represents a specific procedural step in a patent's lifecycle, capturing details such as the step phase, code, result, affected countries, and any relevant time limits. This data is crucial for understanding the procedural aspects of patents, particularly those that are not fully disclosed in the public European Patent Bulletin. Let's dive into the attributes that shape the flow of patent applications within the EP system.

The ``REG201_PROC_STEP`` table plays a fundamental role in documenting the procedural steps a patent application undergoes throughout its lifecycle at the European Patent Office (EPO). Each step is associated with a publication event recorded in the European Patent Bulletin, and the ``BULLETIN_YEAR`` and ``BULLETIN_NR`` fields serve as key timestamps for tracking these events.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG201_PROC_STEP, REG101_APPLN
from sqlalchemy import select, func, case, select, and_

patstat = PatstatClient(env='PROD')

db = patstat.orm()

In [2]:
q = db.query(
    REG201_PROC_STEP.id,
    REG201_PROC_STEP.step_id,
    REG201_PROC_STEP.step_phase,
    REG201_PROC_STEP.step_code,
    REG201_PROC_STEP.step_result,
    REG201_PROC_STEP.step_result_type,
    REG201_PROC_STEP.step_country,
    REG201_PROC_STEP.time_limit,
    REG201_PROC_STEP.time_limit_unit,
    REG201_PROC_STEP.bulletin_year,
    REG201_PROC_STEP.bulletin_nr
).limit(1000)

res = patstat.df(q)
res


,id,step_id,step_phase,step_code,step_result,step_result_type,step_country,time_limit,time_limit_unit,bulletin_year,bulletin_nr
0,13725769,STEP_EXPT_3583591,EXAMN,EXPT,,,,,,0,0
1,7765517,STEP_EXPT_2031902,EXAMN,EXPT,,,,,,0,0
2,14842830,STEP_ADWI_1754300,EXAMN,ADWI,,,,,,2016,39
3,8017026,STEP_EXRE_1037542,EXAMN,EXRE,,,,06,months,0,0
4,5801291,STEP_1412900,REGEN,DEST,,,,,,0,0
...,...,...,...,...,...,...,...,...,...,...,...
995,6801888,STEP_IGRA_492506,EXAMN,IGRA,,,,,,0,0
996,17764820,RENEWAL_9023396,UNDEF,RFEE,,,,,,0,0
997,7705060,STEP_EXPT_2008953,EXAMN,EXPT,,,,,,0,0
998,14177521,STEP_ISAT_2726868,REGEN,ISAT,,,,,,0,0


## Key fields in the REG201_PROC_STEP table

### ID (primary key)
A technical identifier for an application, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [3]:
q = db.query(
    REG201_PROC_STEP.id
).limit(100)

res = patstat.df(q)
res

,id
0,21945856
1,10180079
2,19775110
3,89401907
4,21805774
...,...
95,12819216
96,932625
97,17701749
98,90915446


### STEP_ID (primary key)
The ``STEP_ID`` is a unique identifier assigned to each procedural step within the patent application process. It is present in multiple tables, including ``REG201_PROC_STEP``, ``REG202_PROC_STEP_TEXT``, ``REG203_PROC_STEP_DATE``, ``REG721_PROC_STEP``, ``REG722_PROC_STEP_TEXT``, and ``REG723_PROC_STEP_DATE``. This attribute serves as a key reference for identifying and linking procedural steps across various datasets. With a domain of up to 30 characters, it ensures precise tracking and organization of procedural events associated with patent applications.

### STEP_PHASE
The ``STEP_PHASE`` attribute represents the phase during which a procedural step occurred in the lifecycle of a patent application. Found in the ``REG201_PROC_STEP`` and ``REG721_PROC_STEP`` tables, it categorizes procedural actions into specific stages using standardized codes of up to 5 ASCII characters.

Defaulting to UNDEF if undefined, the possible values include:
– EXAMN: examination phase
– APEXA: appeal in examination
– OPPOS: opposition phase
– APOPP: appeal in opposition
– LIMIT: limitation phase
– REGEN: entry into the regional phase
– INTEX: international examination
– PROPP: petition for review in opposition
– REVOC: revocation phase

In [4]:
q = db.query(
    REG201_PROC_STEP.id,
    func.count(REG201_PROC_STEP.step_id).label("num_steps")
).group_by(REG201_PROC_STEP.id).limit(1000)

res = patstat.df(q)
res


,id,num_steps
0,9711496,3
1,5804123,14
2,963313,3
3,5738928,16
4,90304234,7
...,...,...
995,23175308,9
996,6826745,3
997,17892548,5
998,16164341,20


In the following, we look to the different types of steps.

In [5]:
q = db.query(
    REG201_PROC_STEP.step_phase
).distinct()

res = patstat.df(q)
res

,step_phase
0,REGEN
1,APEXA
2,PROPP
3,UNDEF
4,APOPP
5,EXAMN
6,INTEX
7,LIMIT
8,OPPOS
9,REVOC


Let's count how many occurrences we have for each step phase.

In [6]:
q = db.query(
    REG201_PROC_STEP.step_phase,
    func.count(REG201_PROC_STEP.step_phase).label("phase_count")
).group_by(REG201_PROC_STEP.step_phase)

res = patstat.df(q)
res

,step_phase,phase_count
0,REGEN,16381495
1,APEXA,10863
2,PROPP,96
3,REVOC,1
4,INTEX,900004
5,LIMIT,260
6,UNDEF,15176818
7,OPPOS,389017
8,EXAMN,47189448
9,APOPP,25800


### STEP_CODE
The ``STEP_CODE`` attribute represents a mnemonic identifier for a procedural step in the patent process. This code is essential for categorising and identifying specific actions associated with procedural events. It is found in the ``REG201_PROC_STEP`` and ``REG721_PROC_STEP`` tables and provides a concise alphanumeric representation of each procedural step. This attribute does not have a default value, which allows it to adapt to various scenarios. Codes can be up to 10 characters long, offering flexibility while maintaining clarity. Examples of such codes include OREX and AREX, which correspond to distinct procedural actions. In the 2014 Autumn Edition of the database, 42 unique procedural step codes were identified. The ``STEP_CODE`` ensures a consistent and standardised way to reference procedural steps, enhancing data management and accessibility for users analysing procedural details.

In [7]:
q = db.query(
    REG201_PROC_STEP.step_code
).distinct().limit(15)

res = patstat.df(q)
res

,step_code
0,ADWI
1,DEST
2,EXRE
3,IGRA
4,ORAL
5,TRAN
6,REES
7,OREX
8,REJO
9,OPPC


### STEP_RESULT
The ``STEP_RESULT`` attribute captures the outcome of a procedural step within the patent process. This attribute is included in the ``REG201_PROC_STEP`` and ``REG721_PROC_STEP`` tables and provides valuable information about the resolution or status of specific procedural actions. The attribute's domain allows for a variety of predefined values, such as:
– "yes"
– "no"
– "request accepted"
– "request deemed not to be filed"
– "request granted"
– "request procedure closed"
– "request rejected"
– "request withdrawn"
It also permits an empty string as a valid entry, which serves as the default value when no result is specified. The ``STEP_RESULT`` attribute plays a key role in documenting the procedural journey of a patent application, offering clarity and transparency regarding the decisions made at various stages.

In [8]:
q = db.query(
    REG201_PROC_STEP.step_result
).distinct()

res = patstat.df(q)
res

,step_result
0,yes
1,
2,Request accepted
3,Request deemed not to be filed
4,no
5,Request rejected
6,Request withdrawn
7,Request granted
8,Request procedure closed


### STEP_RESULT_TYPE
The ``STEP_RESULT_TYPE`` attribute defines the type of result associated with a procedural step in the patent process. This attribute is present in the ``REG201_PROC_STEP`` and ``REG721_PROC_STEP`` tables and categorises the result of procedural actions into specific types. The domain for this attribute includes the values "accepted (yes/no)", "RESULT", or an empty string, which serves as the default when no result type is specified. By distinguishing between these types, the ``STEP_RESULT_TYPE`` attribute enhances the granularity of procedural documentation, providing clearer insight into the nature and classification of outcomes in the patent process.

In [9]:
q = db.query(
    REG201_PROC_STEP.step_result_type
).distinct()

res = patstat.df(q)
res

,step_result_type
0,
1,RESULT
2,accepted(yes/no)


In [10]:
q = db.query(
    REG201_PROC_STEP.step_country,
    func.count(REG201_PROC_STEP.step_id).label("steps_per_country")
).group_by(REG201_PROC_STEP.step_country).limit(15)

res = patstat.df(q)
res

,step_country,steps_per_country
0,AU,25862
1,KR,9341
2,FI,1616
3,,79179228
4,EP,470199
5,IN,679
6,PT,1
7,AT,4820
8,FR,1134
9,ES,1576


### TIME_LIMIT
The ``TIME_LIMIT`` attribute specifies a time limit associated with a procedural step in the patent process. Found in the ``REG201_PROC_STEP`` and ``REG721_PROC_STEP`` tables, it provides critical timing information for actions required or deadlines imposed during procedural steps. The domain allows up to 10 characters, with values such as empty (default), numeric indicators (e.g., "01", "06", "12"), or specific formats like "M04" (representing a four-months deadline).

Occasionally, more complex values like "[1986/51]" are used, indicating a deadline set for week 51 of the year 1986. By recording time constraints in procedural steps, the ``TIME_LIMIT`` attribute ensures structured tracking of deadlines, enabling efficient management of time-sensitive tasks in the patent lifecycle.

### TIME_LIMIT_UNIT
The ``TIME_LIMIT_UNIT`` attribute defines the unit of measurement for the time limit associated with a procedural step. It is included in the ``REG201_PROC_STEP`` and ``REG721_PROC_STEP`` tables to provide clarity on the nature of the deadline. The default value is an empty string, indicating no specific unit.

The domain allows up to 6 characters, with "months" being the only commonly used non-default value. This attribute ensures that the duration specified in the corresponding ``TIME_LIMIT`` attribute is interpreted unambiguously, supporting accurate scheduling and adherence to procedural deadlines in the patent process.

### BULLETIN_YEAR 
In the PATSTAT database, the ``BULLETIN_YEAR`` field captures the year when an action or event related to a patent application was published in the European Patent Bulletin. This field plays a critical role in tracking the timeline of patent events, ensuring chronological accuracy in analyses.

The ``BULLETIN_YEAR`` is a 4-digit numeric field (formatted as YYYY), with a default value of 0 to indicate cases where no European Patent Bulletin publication is known. For entries where publication in the European Patent Bulletin is confirmed, ``BULLETIN_YEAR`` reflects the corresponding year of publication. It is used in conjunction with ``BULLETIN_NR``, which specifies the European Patent Bulletin issue number.

### BULLETIN_NR 
The ``BULLETIN_NR`` attribute represents the issue number of the European Patent Bulletin which a specific action has been published. This number indicates the calendar week during which the European Patent Bulletin was released. It serves as a reference for identifying the exact edition of the European Patent Bulletin where actions such as patent grants, publications, or other significant events are announced.

If the action was not published in the European Patent Bulletin or if the information is unknown, the default value of 0 is used for the ``BULLETIN_NR``, which corresponds to the absence of a known European Patent Bulletin number. This value is only used when the associated ``BULLETIN_YEAR`` is also set to 0.

In [11]:
q = db.query(
    REG201_PROC_STEP.id,
    REG201_PROC_STEP.bulletin_year,
    REG201_PROC_STEP.bulletin_nr
).filter(
    REG201_PROC_STEP.bulletin_year != 0
)

res = patstat.df(q)
res

,id,bulletin_year,bulletin_nr
0,6818806,2009,53
1,18893132,2020,53
2,4016903,2009,53
3,13799074,2015,53
4,18891516,2020,53
...,...,...,...
3676831,19956298,2022,52
3676832,6844411,2008,52
3676833,7852802,2009,52
3676834,10250322,2018,52
